In [2]:
# ==========================================
# TASK 1: Data Acquisition & Deep EDA
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create directories for outputs and reports
os.makedirs("data/raw", exist_ok=True)
os.makedirs("docs/screenshots", exist_ok=True)

# 1. Obtain dataset from UCI Repository
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

print("[INFO] Fetching UCI Heart Disease dataset...")
df = pd.read_csv(UCI_URL, names=COLUMNS, na_values='?')

print(f"Original Dataset Shape: {df.shape}")
print("\nMissing Values Check:")
print(df.isnull().sum())

# 2. Data Cleaning: Handle missing values ('?') by dropping rows and binarizing target
df.dropna(inplace=True)
# The original target has classes 0 (no disease) and 1,2,3,4 (presence of disease)
# Convert to binary target: 0 = Normal, 1 = Heart Disease Present
df['target'] = (df['target'] > 0).astype(int)

# Save cleaned dataset
cleaned_path = "data/raw/heart.csv"
df.to_csv(cleaned_path, index=False)
print(f"\n[INFO] Cleaned dataset saved to {cleaned_path}. Final clean shape: {df.shape}")

# 3. Professional EDA Visualizations
sns.set_theme(style="whitegrid")

# Plot 1: Class Balance Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='target', hue='target', data=df, palette='Set2', legend=False)
plt.title("Heart Disease Class Balance Distribution", fontsize=12, fontweight='bold')
plt.xlabel("Target (0: Normal, 1: Heart Disease)", fontsize=10)
plt.ylabel("Patient Count", fontsize=10)
plt.savefig("docs/screenshots/class_balance.png", bbox_inches='tight')
plt.show()

# Plot 2: Correlation Heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5, cbar=True)
plt.title("Feature Correlation Heatmap", fontsize=12, fontweight='bold')
plt.savefig("docs/screenshots/correlation_heatmap.png", bbox_inches='tight')
plt.show()

[INFO] Fetching UCI Heart Disease dataset...
Original Dataset Shape: (303, 14)

Missing Values Check:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64

[INFO] Cleaned dataset saved to data/raw/heart.csv. Final clean shape: (297, 14)


<Figure size 600x400 with 1 Axes>

<Figure size 1000x800 with 2 Axes>

In [3]:
# ==========================================
# TASK 2: Feature Engineering & Modeling
# ==========================================
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

# Load cleaned dataset
df = pd.read_csv("data/raw/heart.csv")
X = df.drop(columns=['target'])
y = df['target']

# Stratified Train-Test Split (80% train, 20% test) to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler artifact for reproducibility (Task 4 requirement)
joblib.dump(scaler, "scaler.pkl")
print("[INFO] StandardScaler fitted and saved as 'scaler.pkl'")

# Define classification models
models = {
    "Logistic_Regression": LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    "Random_Forest": RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
}

# 5-Fold Stratified Cross-Validation & Evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
evaluation_results = []

for name, model in models.items():
    # Cross-validation ROC-AUC scores
    cv_auc_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    
    # Train model on full training set
    model.fit(X_train_scaled, y_train)
    
    # Predict on test set
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    evaluation_results.append({
        "Model": name,
        "CV ROC-AUC Mean": round(cv_auc_scores.mean(), 4),
        "CV ROC-AUC Std": round(cv_auc_scores.std(), 4),
        "Test Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "ROC-AUC": round(auc, 4)
    })
    
    # Save the Random Forest model as primary deployment artifact
    if name == "Random_Forest":
        joblib.dump(model, "model.pkl")

# Display evaluation summary table
results_df = pd.DataFrame(evaluation_results)
print("\n--- Model Evaluation Summary ---")
display(results_df)

[INFO] StandardScaler fitted and saved as 'scaler.pkl'

--- Model Evaluation Summary ---


,Model,CV ROC-AUC Mean,CV ROC-AUC Std,Test Accuracy,Precision,Recall,ROC-AUC
0,Logistic_Regression,0.8755,0.0664,0.85,0.8800,0.7857,0.9520
1,Random_Forest,0.8827,0.0661,0.80,0.8077,0.7500,0.9286


In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline

print("[INFO] Tuning and boosting models...")

# 1. Hyperparameter Tuning for Random Forest using GridSearchCV
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8, 10],
    'min_samples_split': [2, 5, 10]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42), 
    rf_params, 
    cv=5, 
    scoring='roc_auc', 
    n_jobs=-1
)
rf_grid.fit(X_train_scaled, y_train)
best_rf = rf_grid.best_estimator_

print(f"Best Random Forest Parameters: {rf_grid.best_params_}")

# 2. Introduce Gradient Boosting (often performs exceptionally well on UCI Heart dataset)
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
gb_model.fit(X_train_scaled, y_train)

# Evaluate Tuned Random Forest & Gradient Boosting
boosted_models = {
    "Tuned_Random_Forest": best_rf,
    "Gradient_Boosting": gb_model
}

boosted_results = []

for name, model in boosted_models.items():
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    boosted_results.append({
        "Model": name,
        "Test Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "ROC-AUC": round(auc, 4)
    })

# Update primary deployment model to the best performing one
if roc_auc_score(y_test, best_rf.predict_proba(X_test_scaled)[:, 1]) > roc_auc_score(y_test, gb_model.predict_proba(X_test_scaled)[:, 1]):
    joblib.dump(best_rf, "model.pkl")
    print("\n[SUCCESS] Tuned Random Forest saved as primary 'model.pkl'.")
else:
    joblib.dump(gb_model, "model.pkl")
    print("\n[SUCCESS] Gradient Boosting saved as primary 'model.pkl'.")

boosted_df = pd.DataFrame(boosted_results)
display(boosted_df)

[INFO] Tuning and boosting models...
Best Random Forest Parameters: {'max_depth': 6, 'min_samples_split': 2, 'n_estimators': 300}

[SUCCESS] Tuned Random Forest saved as primary 'model.pkl'.


,Model,Test Accuracy,Precision,Recall,ROC-AUC
0,Tuned_Random_Forest,0.8333,0.875,0.75,0.9342
1,Gradient_Boosting,0.8167,0.840,0.75,0.9129


In [5]:
# ==========================================
# TASK 3 & 4: MLflow Tracking & Artifact Packaging
# ==========================================
%pip install --upgrade "setuptools<70.0.0"

import mlflow
import mlflow.sklearn

# Set MLflow experiment name
mlflow.set_experiment("BITS_MLOps_Heart_Disease_Tuned")

# Dictionary of all models we want to track
final_models_to_track = {
    "Tuned_Random_Forest": best_rf,
    "Gradient_Boosting": gb_model
}

for name, model in final_models_to_track.items():
    with mlflow.start_run(run_name=name):
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
        
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds)
        rec = recall_score(y_test, preds)
        auc = roc_auc_score(y_test, probs)
        
        # Log parameters, metrics, and models to MLflow
        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            "test_accuracy": acc,
            "precision": prec,
            "recall": rec,
            "roc_auc": auc
        })
        mlflow.sklearn.log_model(model, artifact_path=name)
        
        print(f"[MLflow] Successfully logged and tracked: {name} (ROC-AUC: {auc:.4f})")

print("\n[SUCCESS] Task 3 (Experiment Tracking) and Task 4 (Model Packaging) are fully completed!")

Note: you may need to restart the kernel to use updated packages.
[MLflow] Successfully logged and tracked: Tuned_Random_Forest (ROC-AUC: 0.9342)
[MLflow] Successfully logged and tracked: Gradient_Boosting (ROC-AUC: 0.9129)

[SUCCESS] Task 3 (Experiment Tracking) and Task 4 (Model Packaging) are fully completed!


In [6]:
'''
# ==========================================
# TASK 5: Unit Testing setup via Jupyter
# ==========================================
import os

# Create tests directory
os.makedirs("tests", exist_ok=True)

# Write the test file using %%writefile
%%writefile tests/test_api.py
import os
import pandas as pd

def test_dataset_exists():
    """Verify that the cleaned dataset exists and is non-empty."""
    assert os.path.exists("data/raw/heart.csv"), "Cleaned dataset not found!"
    df = pd.read_csv("data/raw/heart.csv")
    assert not df.empty, "Dataset is empty!"
    assert 'target' in df.columns, "Target column missing from dataset!"

def test_model_artifacts_exist():
    """Verify that model and scaler artifacts are successfully saved."""
    assert os.path.exists("model.pkl"), "Trained model artifact (model.pkl) missing!"
    assert os.path.exists("scaler.pkl"), "Scaler artifact
''' (scaler.pkl) missing!"

UsageError: Line magic function `%%writefile` not found.


In [7]:
import os
os.makedirs("tests", exist_ok=True)

In [8]:
import os
import pandas as pd

def test_dataset_exists():
    """Verify that the cleaned dataset exists and is non-empty."""
    assert os.path.exists("data/raw/heart.csv"), "Cleaned dataset not found!"
    df = pd.read_csv("data/raw/heart.csv")
    assert not df.empty, "Dataset is empty!"
    assert 'target' in df.columns, "Target column missing from dataset!"

def test_model_artifacts_exist():
    """Verify that model and scaler artifacts are successfully saved."""
    assert os.path.exists("model.pkl"), "Trained model artifact (model.pkl) missing!"
    assert os.path.exists("scaler.pkl"), "Scaler artifact (scaler.pkl) missing!"

In [9]:
!pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.0.0, pluggy-1.7.0 -- /home/s2025af05011/workspace/heart-disease-mlops/venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/s2025af05011/workspace/heart-disease-mlops
plugins: platformdirs-4.13.4, anyio-4.15.1
collected 3 items                                                              

tests/test_api.py::test_dataset_exists PASSED                            [ 33%]
tests/test_api.py::test_model_artifacts_exist PASSED                     [ 66%]
tests/test_api.py::test_model_inference PASSED                           [100%]

=============================== warnings summary ===============================
tests/test_api.py::test_model_inference
  /home/s2025af05011/workspace/heart-disease-mlops/venv/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
    warnings.warn

In [10]:
!pytest tests/test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.0.0, pluggy-1.7.0 -- /home/s2025af05011/workspace/heart-disease-mlops/venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/s2025af05011/workspace/heart-disease-mlops
plugins: platformdirs-4.13.4, anyio-4.15.1
collected 3 items                                                              

tests/test_api.py::test_dataset_exists PASSED                            [ 33%]
tests/test_api.py::test_model_artifacts_exist PASSED                     [ 66%]
tests/test_api.py::test_model_inference PASSED                           [100%]

=============================== warnings summary ===============================
tests/test_api.py::test_model_inference
  /home/s2025af05011/workspace/heart-disease-mlops/venv/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
    warnings.warn

In [11]:
import os

# Ensure the tests directory exists
os.makedirs("tests", exist_ok=True)

# Write the test file explicitly using Python I/O
test_code = """import os
import pandas as pd

def test_dataset_exists():
    \"\"\"Verify that the cleaned dataset exists and is non-empty.\"\"\"
    assert os.path.exists("data/raw/heart.csv"), "Cleaned dataset not found!"
    df = pd.read_csv("data/raw/heart.csv")
    assert not df.empty, "Dataset is empty!"
    assert 'target' in df.columns, "Target column missing from dataset!"

def test_model_artifacts_exist():
    \"\"\"Verify that model and scaler artifacts are successfully saved.\"\"\"
    assert os.path.exists("model.pkl"), "Trained model artifact (model.pkl) missing!"
    assert os.path.exists("scaler.pkl"), "Scaler artifact (scaler.pkl) missing!"
"""

with open("tests/test_api.py", "w") as f:
    f.write(test_code)

print("[INFO] Successfully created tests/test_api.py")

[INFO] Successfully created tests/test_api.py


In [12]:
!pytest tests/test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.0.0, pluggy-1.7.0 -- /home/s2025af05011/workspace/heart-disease-mlops/venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/s2025af05011/workspace/heart-disease-mlops
plugins: platformdirs-4.13.4, anyio-4.15.1
collected 2 items                                                              

tests/test_api.py::test_dataset_exists PASSED                            [ 50%]
tests/test_api.py::test_model_artifacts_exist PASSED                     [100%]

============================== 2 passed in 1.16s ===============================


In [13]:
import os

os.makedirs("tests", exist_ok=True)

expanded_test_code = """import os
import joblib
import numpy as np
import pandas as pd

def test_dataset_exists():
    \"\"\"Verify that the cleaned dataset exists and is non-empty.\"\"\"
    assert os.path.exists("data/raw/heart.csv"), "Cleaned dataset not found!"
    df = pd.read_csv("data/raw/heart.csv")
    assert not df.empty, "Dataset is empty!"
    assert 'target' in df.columns, "Target column missing from dataset!"

def test_model_artifacts_exist():
    \"\"\"Verify that model and scaler artifacts are successfully saved.\"\"\"
    assert os.path.exists("model.pkl"), "Trained model artifact (model.pkl) missing!"
    assert os.path.exists("scaler.pkl"), "Scaler artifact (scaler.pkl) missing!"

def test_model_inference():
    \"\"\"Verify that the saved model and scaler can perform a prediction successfully.\"\"\"
    model = joblib.load("model.pkl")
    scaler = joblib.load("scaler.pkl")
    
    dummy_input = np.array([[63.0, 1.0, 3.0, 145.0, 233.0, 1.0, 0.0, 150.0, 0.0, 2.3, 0.0, 0.0, 1.0]])
    
    scaled_input = scaler.transform(dummy_input)
    prediction = model.predict(scaled_input)
    probabilities = model.predict_proba(scaled_input)
    
    assert len(prediction) == 1, "Prediction output length mismatch!"
    assert prediction[0] in [0, 1], "Prediction should be binary (0 or 1)!"
    assert probabilities.shape == (1, 2), "Probability shape should be (1, 2)!"
"""

with open("tests/test_api.py", "w") as f:
    f.write(expanded_test_code)

print("[INFO] Expanded unit tests written successfully.")

[INFO] Expanded unit tests written successfully.


In [14]:
!pytest tests/test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.0.0, pluggy-1.7.0 -- /home/s2025af05011/workspace/heart-disease-mlops/venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/s2025af05011/workspace/heart-disease-mlops
plugins: platformdirs-4.13.4, anyio-4.15.1
collected 3 items                                                              

tests/test_api.py::test_dataset_exists PASSED                            [ 33%]
tests/test_api.py::test_model_artifacts_exist PASSED                     [ 66%]
tests/test_api.py::test_model_inference PASSED                           [100%]

=============================== warnings summary ===============================
tests/test_api.py::test_model_inference
  /home/s2025af05011/workspace/heart-disease-mlops/venv/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
    warnings.warn

In [15]:
import os

os.makedirs("src", exist_ok=True)

api_code = """from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import numpy as np
import pandas as pd

app = FastAPI(title="Heart Disease Prediction API", version="1.0.0")

try:
    scaler = joblib.load("scaler.pkl")
    model = joblib.load("model.pkl")
except Exception as e:
    scaler, model = None, None

class PatientPayload(BaseModel):
    age: float = Field(..., example=63.0)
    sex: float = Field(..., example=1.0)
    cp: float = Field(..., example=3.0)
    trestbps: float = Field(..., example=145.0)
    chol: float = Field(..., example=233.0)
    fbs: float = Field(..., example=1.0)
    restecg: float = Field(..., example=0.0)
    thalach: float = Field(..., example=150.0)
    exang: float = Field(..., example=0.0)
    oldpeak: float = Field(..., example=2.3)
    slope: float = Field(..., example=0.0)
    ca: float = Field(..., example=0.0)
    thal: float = Field(..., example=1.0)

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict")
def predict(patient: PatientPayload):
    if not model or not scaler:
        raise HTTPException(status_code=500, detail="Model or scaler artifacts missing.")
    
    feature_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
    input_df = pd.DataFrame([[
        patient.age, patient.sex, patient.cp, patient.trestbps, patient.chol,
        patient.fbs, patient.restecg, patient.thalach, patient.exang,
        patient.oldpeak, patient.slope, patient.ca, patient.thal
    ]], columns=feature_names)
    
    scaled = scaler.transform(input_df)
    pred = int(model.predict(scaled)[0])
    prob = float(np.max(model.predict_proba(scaled)))
    
    return {
        "prediction": pred,
        "diagnosis": "High Risk of Heart Disease" if pred == 1 else "Normal",
        "confidence": round(prob, 4)
    }
"""

with open("src/app.py", "w") as f:
    f.write(api_code)

print("Successfully created src/app.py")

Successfully created src/app.py


In [16]:
dockerfile_content = """FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip
RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY scaler.pkl model.pkl ./

EXPOSE 8000

CMD ["uvicorn", "src.app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("[INFO] Successfully created Dockerfile")

[INFO] Successfully created Dockerfile


In [17]:
!pip install --upgrade fastapi starlette

In [18]:
from fastapi.testclient import TestClient
from src.app import app

client = TestClient(app)

# Test health endpoint
response = client.get("/health")
print("Health Check Response:", response.json())

# Test predict endpoint with sample patient data
payload = {
    "age": 63.0, "sex": 1.0, "cp": 3.0, "trestbps": 145.0, "chol": 233.0,
    "fbs": 1.0, "restecg": 0.0, "thalach": 150.0, "exang": 0.0,
    "oldpeak": 2.3, "slope": 0.0, "ca": 0.0, "thal": 1.0
}
response = client.post("/predict", json=payload)
print("Prediction Response:", response.json())

Health Check Response: {'status': 'healthy', 'model_loaded': True}
Prediction Response: {'prediction': 0, 'diagnosis': 'Normal', 'confidence': 0.7814}


In [19]:
dockerfile_content = """FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip
RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY scaler.pkl model.pkl ./

EXPOSE 8000

CMD ["uvicorn", "src.app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

print("[INFO] Successfully created Dockerfile")

[INFO] Successfully created Dockerfile


In [20]:
import os

# Create the .github/workflows directory
os.makedirs(".github/workflows", exist_ok=True)

ci_yaml_content = """name: MLOps CI/CD Pipeline

on:
  push:
    branches: [ main, master ]
  pull_request:
    branches: [ main, master ]

jobs:
  build-and-test:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout code
      uses: actions/checkout@v4

    - name: Set up Python 3.12
      uses: actions/setup-python@v5
      with:
        python-version: '3.12'
        cache: 'pip'

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt

    - name: Run Unit Tests with Pytest
      run: |
        pytest tests/ -v
"""

with open(".github/workflows/ci.yml", "w") as f:
    f.write(ci_yaml_content)

print("[INFO] Successfully created .github/workflows/ci.yml")

[INFO] Successfully created .github/workflows/ci.yml


In [21]:
import os

# Create the k8s directory
os.makedirs("k8s", exist_ok=True)

k8s_yaml_content = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: heart-disease-api-deployment
  labels:
    app: heart-disease-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: heart-disease-api
  template:
    metadata:
      labels:
        app: heart-disease-api
    spec:
      containers:
      - name: heart-disease-api
        image: your-dockerhub-username/heart-disease-api:latest  # Update with your container registry image
        ports:
        - containerPort: 8000
        resources:
          limits:
            cpu: "500m"
            memory: "512Mi"
          requests:
            cpu: "200m"
            memory: "256Mi"
        readinessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 5
          periodSeconds: 10
        livenessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 10
          periodSeconds: 15

---
apiVersion: v1
kind: Service
metadata:
  name: heart-disease-api-service
spec:
  type: LoadBalancer
  selector:
    app: heart-disease-api
  ports:
    - protocol: TCP
      port: 80
      targetPort: 8000
"""

with open("k8s/deployment.yaml", "w") as f:
    f.write(k8s_yaml_content)

print("[INFO] Successfully created k8s/deployment.yaml")

[INFO] Successfully created k8s/deployment.yaml


In [22]:
import os

# Create the k8s directory
os.makedirs("k8s", exist_ok=True)

k8s_yaml_content = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: heart-disease-api-deployment
  labels:
    app: heart-disease-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: heart-disease-api
  template:
    metadata:
      labels:
        app: heart-disease-api
    spec:
      containers:
      - name: heart-disease-api
        image: your-dockerhub-username/heart-disease-api:latest  # Update with your container registry image
        ports:
        - containerPort: 8000
        resources:
          limits:
            cpu: "500m"
            memory: "512Mi"
          requests:
            cpu: "200m"
            memory: "256Mi"
        readinessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 5
          periodSeconds: 10
        livenessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 10
          periodSeconds: 15

---
apiVersion: v1
kind: Service
metadata:
  name: heart-disease-api-service
spec:
  type: LoadBalancer
  selector:
    app: heart-disease-api
  ports:
    - protocol: TCP
      port: 80
      targetPort: 8000
"""

with open("k8s/deployment.yaml", "w") as f:
    f.write(k8s_yaml_content)

print("[INFO] Successfully created k8s/deployment.yaml")

[INFO] Successfully created k8s/deployment.yaml


In [23]:
import os
import time
import logging
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field
import joblib
import numpy as np
import pandas as pd

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

app = FastAPI(title="Heart Disease Prediction API", version="1.0.0")

# Middleware for request logging & basic monitoring metrics
@app.middleware("http")
async def log_requests(request: Request, call_next):
    start_time = time.time()
    response = await call_next(request)
    process_time = (time.time() - start_time) * 1000
    logger.info(f"Method: {request.method} Path: {request.url.path} Status: {response.status_code} Duration: {process_time:.2f}ms")
    return response

# Load model and scaler at startup
try:
    scaler = joblib.load("scaler.pkl")
    model = joblib.load("model.pkl")
except Exception as e:
    scaler, model = None, None

class PatientPayload(BaseModel):
    age: float = Field(..., example=63.0)
    sex: float = Field(..., example=1.0)
    cp: float = Field(..., example=3.0)
    trestbps: float = Field(..., example=145.0)
    chol: float = Field(..., example=233.0)
    fbs: float = Field(..., example=1.0)
    restecg: float = Field(..., example=0.0)
    thalach: float = Field(..., example=150.0)
    exang: float = Field(..., example=0.0)
    oldpeak: float = Field(..., example=2.3)
    slope: float = Field(..., example=0.0)
    ca: float = Field(..., example=0.0)
    thal: float = Field(..., example=1.0)

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict")
def predict(patient: PatientPayload):
    if not model or not scaler:
        raise HTTPException(status_code=500, detail="Model or scaler artifacts missing.")
    
    feature_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
    input_df = pd.DataFrame([[
        patient.age, patient.sex, patient.cp, patient.trestbps, patient.chol,
        patient.fbs, patient.restecg, patient.thalach, patient.exang,
        patient.oldpeak, patient.slope, patient.ca, patient.thal
    ]], columns=feature_names)
    
    scaled = scaler.transform(input_df)
    pred = int(model.predict(scaled)[0])
    prob = float(np.max(model.predict_proba(scaled)))
    
    return {
        "prediction": pred,
        "diagnosis": "High Risk of Heart Disease" if pred == 1 else "Normal",
        "confidence": round(prob, 4)
    }
"""

with open("src/app.py", "w") as f:
    f.write(api_code)

print("[INFO] Successfully updated src/app.py with logging middleware.")

SyntaxError: incomplete input (4286114120.py, line 72)

In [24]:
import threading
import uvicorn
from src.app import app

def run_server():
    # Run uvicorn server on port 8000
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Start the server in a background thread so the notebook doesn't freeze
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("[INFO] FastAPI server started in background on http://127.0.0.1:8000")

[INFO] FastAPI server started in background on http://127.0.0.1:8000


INFO:     Started server process [7532]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [25]:
import threading
import uvicorn
from src.app import app

def run_server():
    # Run uvicorn server on port 8000
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Start the server in a background thread so the notebook doesn't freeze
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("[INFO] FastAPI server started in background on http://127.0.0.1:8000")

[INFO] FastAPI server started in background on http://127.0.0.1:8000


INFO:     Started server process [7532]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [26]:
import threading
import uvicorn
import requests
from src.app import app

# 1. Function to run Uvicorn on a free port (e.g., 8001 to avoid port 8000 conflicts)
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="info")

# 2. Start the server in a background thread so the notebook doesn't block
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("[INFO] FastAPI server started in background on http://127.0.0.1:8001")

# 3. Pause briefly to ensure the server has fully initialized
import time
time.sleep(2)

# 4. Test the Health Endpoint
health_res = requests.get("http://127.0.0.1:8001/health")
print("\n--- Health Check ---")
print(health_res.json())

# 5. Test the Prediction Endpoint
payload = {
    "age": 63.0, "sex": 1.0, "cp": 3.0, "trestbps": 145.0, "chol": 233.0,
    "fbs": 1.0, "restecg": 0.0, "thalach": 150.0, "exang": 0.0,
    "oldpeak": 2.3, "slope": 0.0, "ca": 0.0, "thal": 1.0
}
pred_res = requests.post("http://127.0.0.1:8001/predict", json=payload)
print("\n--- Prediction Response ---")
print(pred_res.json())

# 6. Fetch Prometheus Metrics
metrics_res = requests.get("http://127.0.0.1:8001/metrics")
print("\n--- Prometheus Metrics Sample ---")
print("\n".join(metrics_res.text.splitlines()[:12]))

[INFO] FastAPI server started in background on http://127.0.0.1:8001

INFO:     Started server process [7532]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)



INFO:     127.0.0.1:51660 - "GET /health HTTP/1.1" 200 OK

--- Health Check ---
{'status': 'healthy', 'model_loaded': True}
INFO:     127.0.0.1:51670 - "POST /predict HTTP/1.1" 200 OK

--- Prediction Response ---
{'prediction': 0, 'diagnosis': 'Normal', 'confidence': 0.7814}
INFO:     127.0.0.1:51680 - "GET /metrics HTTP/1.1" 404 Not Found

--- Prometheus Metrics Sample ---
{"detail":"Not Found"}


In [28]:
import os

api_code = """import os
import time
import logging
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field
import joblib
import numpy as np
import pandas as pd
from prometheus_fastapi_instrumentator import Instrumentator

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

app = FastAPI(title="Heart Disease Prediction API", version="1.0.0")

# Initialize and instrument Prometheus metrics
instrumentator = Instrumentator()
instrumentator.instrument(app)

# Middleware for request logging
@app.middleware("http")
async def log_requests(request: Request, call_next):
    start_time = time.time()
    response = await call_next(request)
    process_time = (time.time() - start_time) * 1000
    logger.info(f"Method: {request.method} Path: {request.url.path} Status: {response.status_code} Duration: {process_time:.2f}ms")
    return response

# Expose the /metrics endpoint on startup
@app.on_event("startup")
async def startup_event():
    instrumentator.expose(app, endpoint="/metrics")

# Load model and scaler at startup
try:
    scaler = joblib.load("scaler.pkl")
    model = joblib.load("model.pkl")
except Exception as e:
    scaler, model = None, None

class PatientPayload(BaseModel):
    age: float = Field(..., example=63.0)
    sex: float = Field(..., example=1.0)
    cp: float = Field(..., example=3.0)
    trestbps: float = Field(..., example=145.0)
    chol: float = Field(..., example=233.0)
    fbs: float = Field(..., example=1.0)
    restecg: float = Field(..., example=0.0)
    thalach: float = Field(..., example=150.0)
    exang: float = Field(..., example=0.0)
    oldpeak: float = Field(..., example=2.3)
    slope: float = Field(..., example=0.0)
    ca: float = Field(..., example=0.0)
    thal: float = Field(..., example=1.0)

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict")
def predict(patient: PatientPayload):
    if not model or not scaler:
        raise HTTPException(status_code=500, detail="Model or scaler artifacts missing.")
    
    feature_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
    input_df = pd.DataFrame([[
        patient.age, patient.sex, patient.cp, patient.trestbps, patient.chol,
        patient.fbs, patient.restecg, patient.thalach, patient.exang,
        patient.oldpeak, patient.slope, patient.ca, patient.thal
    ]], columns=feature_names)
    
    scaled = scaler.transform(input_df)
    pred = int(model.predict(scaled)[0])
    prob = float(np.max(model.predict_proba(scaled)))
    
    return {
        "prediction": pred,
        "diagnosis": "High Risk of Heart Disease" if pred == 1 else "Normal",
        "confidence": round(prob, 4)
    }
"""

with open("src/app.py", "w") as f:
    f.write(api_code)

print("[INFO] Successfully updated src/app.py with proper Prometheus instrumentation.")ut


SyntaxError: invalid syntax (1538551741.py, line 89)

In [29]:
import os

api_code = """import os
import time
import logging
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field
import joblib
import numpy as np
import pandas as pd
from prometheus_fastapi_instrumentator import Instrumentator

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

app = FastAPI(title="Heart Disease Prediction API", version="1.0.0")

# Initialize and instrument Prometheus metrics
instrumentator = Instrumentator()
instrumentator.instrument(app)

# Middleware for request logging
@app.middleware("http")
async def log_requests(request: Request, call_next):
    start_time = time.time()
    response = await call_next(request)
    process_time = (time.time() - start_time) * 1000
    logger.info(f"Method: {request.method} Path: {request.url.path} Status: {response.status_code} Duration: {process_time:.2f}ms")
    return response

# Expose the /metrics endpoint on startup
@app.on_event("startup")
async def startup_event():
    instrumentator.expose(app, endpoint="/metrics")

# Load model and scaler at startup
try:
    scaler = joblib.load("scaler.pkl")
    model = joblib.load("model.pkl")
except Exception as e:
    scaler, model = None, None

class PatientPayload(BaseModel):
    age: float = Field(..., example=63.0)
    sex: float = Field(..., example=1.0)
    cp: float = Field(..., example=3.0)
    trestbps: float = Field(..., example=145.0)
    chol: float = Field(..., example=233.0)
    fbs: float = Field(..., example=1.0)
    restecg: float = Field(..., example=0.0)
    thalach: float = Field(..., example=150.0)
    exang: float = Field(..., example=0.0)
    oldpeak: float = Field(..., example=2.3)
    slope: float = Field(..., example=0.0)
    ca: float = Field(..., example=0.0)
    thal: float = Field(..., example=1.0)

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict")
def predict(patient: PatientPayload):
    if not model or not scaler:
        raise HTTPException(status_code=500, detail="Model or scaler artifacts missing.")
    
    feature_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
    input_df = pd.DataFrame([[
        patient.age, patient.sex, patient.cp, patient.trestbps, patient.chol,
        patient.fbs, patient.restecg, patient.thalach, patient.exang,
        patient.oldpeak, patient.slope, patient.ca, patient.thal
    ]], columns=feature_names)
    
    scaled = scaler.transform(input_df)
    pred = int(model.predict(scaled)[0])
    prob = float(np.max(model.predict_proba(scaled)))
    
    return {
        "prediction": pred,
        "diagnosis": "High Risk of Heart Disease" if pred == 1 else "Normal",
        "confidence": round(prob, 4)
    }
"""

with open("src/app.py", "w") as f:
    f.write(api_code)

print("[INFO] Successfully updated src/app.py with proper Prometheus instrumentation.")

[INFO] Successfully updated src/app.py with proper Prometheus instrumentation.


In [32]:
import threading
import uvicorn
import requests
import time
from src.app import app

# 1. Start the server on port 8001
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # Wait for startup

# 2. Test Endpoints
health_res = requests.get("http://127.0.0.1:8001/health")
print("Health Check:", health_res.json())

payload = {
    "age": 63.0, "sex": 1.0, "cp": 3.0, "trestbps": 145.0, "chol": 233.0,
    "fbs": 1.0, "restecg": 0.0, "thalach": 150.0, "exang": 0.0,
    "oldpeak": 2.3, "slope": 0.0, "ca": 0.0, "thal": 1.0
}
pred_res = requests.post("http://127.0.0.1:8001/predict", json=payload)
print("Prediction Response:", pred_res.json())

# 3. Fetch Prometheus Metrics
metrics_res = requests.get("http://127.0.0.1:8001/metrics")
print(f"Metrics Endpoint Status: {metrics_res.status_code}")
print("\n--- Prometheus Metrics Sample ---")
print("\n".join(metrics_res.text.splitlines()[:10]))

INFO:     Started server process [7532]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8001): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:57660 - "GET /health HTTP/1.1" 200 OK
Health Check: {'status': 'healthy', 'model_loaded': True}
INFO:     127.0.0.1:57670 - "POST /predict HTTP/1.1" 200 OK
Prediction Response: {'prediction': 0, 'diagnosis': 'Normal', 'confidence': 0.7814}
INFO:     127.0.0.1:57684 - "GET /metrics HTTP/1.1" 404 Not Found
Metrics Endpoint Status: 404

--- Prometheus Metrics Sample ---
{"detail":"Not Found"}


In [31]:
import os

api_code = """import os
import time
import logging
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field
import joblib
import numpy as np
import pandas as pd
from prometheus_fastapi_instrumentator import Instrumentator

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

app = FastAPI(title="Heart Disease Prediction API", version="1.0.0")

# Instrument and immediately expose Prometheus metrics on /metrics
Instrumentator().instrument(app).expose(app, endpoint="/metrics")

# Middleware for request logging
@app.middleware("http")
async def log_requests(request: Request, call_next):
    start_time = time.time()
    response = await call_next(request)
    process_time = (time.time() - start_time) * 1000
    logger.info(f"Method: {request.method} Path: {request.url.path} Status: {response.status_code} Duration: {process_time:.2f}ms")
    return response

# Load model and scaler at startup
try:
    scaler = joblib.load("scaler.pkl")
    model = joblib.load("model.pkl")
except Exception as e:
    scaler, model = None, None

class PatientPayload(BaseModel):
    age: float = Field(..., example=63.0)
    sex: float = Field(..., example=1.0)
    cp: float = Field(..., example=3.0)
    trestbps: float = Field(..., example=145.0)
    chol: float = Field(..., example=233.0)
    fbs: float = Field(..., example=1.0)
    restecg: float = Field(..., example=0.0)
    thalach: float = Field(..., example=150.0)
    exang: float = Field(..., example=0.0)
    oldpeak: float = Field(..., example=2.3)
    slope: float = Field(..., example=0.0)
    ca: float = Field(..., example=0.0)
    thal: float = Field(..., example=1.0)

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict")
def predict(patient: PatientPayload):
    if not model or not scaler:
        raise HTTPException(status_code=500, detail="Model or scaler artifacts missing.")
    
    feature_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
    input_df = pd.DataFrame([[
        patient.age, patient.sex, patient.cp, patient.trestbps, patient.chol,
        patient.fbs, patient.restecg, patient.thalach, patient.exang,
        patient.oldpeak, patient.slope, patient.ca, patient.thal
    ]], columns=feature_names)
    
    scaled = scaler.transform(input_df)
    pred = int(model.predict(scaled)[0])
    prob = float(np.max(model.predict_proba(scaled)))
    
    return {
        "prediction": pred,
        "diagnosis": "High Risk of Heart Disease" if pred == 1 else "Normal",
        "confidence": round(prob, 4)
    }
"""

with open("src/app.py", "w") as f:
    f.write(api_code)

print("[INFO] Successfully updated src/app.py with inline Prometheus exposition.")

[INFO] Successfully updated src/app.py with inline Prometheus exposition.


In [34]:
import threading
import uvicorn
import requests
import time
import importlib
import src.app
from src.app import app

# Force Python to reload the newly updated app module from disk
importlib.reload(src.app)
from src.app import app

# 1. Start the server on port 8002
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8002, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # Wait for startup

# 2. Test Endpoints
health_res = requests.get("http://127.0.0.1:8002/health")
print("Health Check:", health_res.json())

payload = {
    "age": 63.0, "sex": 1.0, "cp": 3.0, "trestbps": 145.0, "chol": 233.0,
    "fbs": 1.0, "restecg": 0.0, "thalach": 150.0, "exang": 0.0,
    "oldpeak": 2.3, "slope": 0.0, "ca": 0.0, "thal": 1.0
}
pred_res = requests.post("http://127.0.0.1:8002/predict", json=payload)
print("Prediction Response:", pred_res.json())

# 3. Fetch Prometheus Metrics
metrics_res = requests.get("http://127.0.0.1:8002/metrics")
print(f"Metrics Endpoint Status: {metrics_res.status_code}")
print("\n--- Prometheus Metrics Sample ---")
print("\n".join(metrics_res.text.splitlines()[:12]))

INFO:     Started server process [7532]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8002): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
2026-10-18 20:02:36,574 - INFO - Method: GET Path: /health Status: 200 Duration: 1.02ms


INFO:     127.0.0.1:52234 - "GET /health HTTP/1.1" 200 OK


2026-10-18 20:02:36,622 - INFO - Method: POST Path: /predict Status: 200 Duration: 41.13ms


Health Check: {'status': 'healthy', 'model_loaded': True}
INFO:     127.0.0.1:52236 - "POST /predict HTTP/1.1" 200 OK


2026-10-18 20:02:36,632 - INFO - Method: GET Path: /metrics Status: 200 Duration: 2.80ms


Prediction Response: {'prediction': 0, 'diagnosis': 'Normal', 'confidence': 0.7814}
INFO:     127.0.0.1:52238 - "GET /metrics HTTP/1.1" 200 OK
Metrics Endpoint Status: 200

--- Prometheus Metrics Sample ---
# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 19588.0
python_gc_objects_collected_total{generation="1"} 3982.0
python_gc_objects_collected_total{generation="2"} 19579.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter


In [36]:
import threading
import uvicorn
import requests
import time
import importlib
import src.app
from src.app import app

# Force Python to reload the newly updated app module from disk
importlib.reload(src.app)
from src.app import app

# 1. Start the server on port 8002
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8002, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # Wait for startup

# 2. Test Endpoints
health_res = requests.get("http://127.0.0.1:8002/health")
print("Health Check:", health_res.json())

payload = {
    "age": 63.0, "sex": 1.0, "cp": 3.0, "trestbps": 145.0, "chol": 233.0,
    "fbs": 1.0, "restecg": 0.0, "thalach": 150.0, "exang": 0.0,
    "oldpeak": 2.3, "slope": 0.0, "ca": 0.0, "thal": 1.0
}
pred_res = requests.post("http://127.0.0.1:8002/predict", json=payload)
print("Prediction Response:", pred_res.json())

# 3. Fetch Prometheus Metrics
metrics_res = requests.get("http://127.0.0.1:8002/metrics")
print(f"Metrics Endpoint Status: {metrics_res.status_code}")
print("\n--- Prometheus Metrics Sample ---")
print("\n".join(metrics_res.text.splitlines()[:12]))

INFO:     Started server process [7532]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8002): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
2026-10-18 20:04:29,524 - INFO - Method: GET Path: /health Status: 200 Duration: 1.00ms


INFO:     127.0.0.1:47666 - "GET /health HTTP/1.1" 200 OK


2026-10-18 20:04:29,579 - INFO - Method: POST Path: /predict Status: 200 Duration: 47.56ms


Health Check: {'status': 'healthy', 'model_loaded': True}
INFO:     127.0.0.1:47682 - "POST /predict HTTP/1.1" 200 OK


2026-10-18 20:04:29,589 - INFO - Method: GET Path: /metrics Status: 200 Duration: 3.36ms


Prediction Response: {'prediction': 0, 'diagnosis': 'Normal', 'confidence': 0.7814}
INFO:     127.0.0.1:47688 - "GET /metrics HTTP/1.1" 200 OK
Metrics Endpoint Status: 200

--- Prometheus Metrics Sample ---
# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 19798.0
python_gc_objects_collected_total{generation="1"} 3982.0
python_gc_objects_collected_total{generation="2"} 19579.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
